# 10. AIND Ecephys NWB Export

Run [aind-ecephys-nwb](https://github.com/AllenNeuralDynamics/aind-ecephys-nwb/tree/main/code) on the dispatched recording from `1_aind_ephys_dispatch.ipynb` to produce an NWB file.

The capsule reads `data/job_*.json` (the dispatcher output) to find each raw recording, then writes one NWB per (block, recording) pair. We pass `--write-raw` to include the raw `ElectricalSeries` and `--skip-lfp` (the toy data has no separate LFP stream and modern spikeinterface's `highpass_check` rejects the capsule's hard-coded LFP `freq_min=0.5`).

## Imports and deps

In [1]:
import json
import shutil
import subprocess
import sys
from pathlib import Path

In [2]:
subprocess.run(
    [
        "uv", "pip", "install", "--python", sys.executable,
        "neuroconv", "pynwb", "hdmf-zarr", "aind-nwb-utils",
    ],
    check=True,
)

Using Python 3.12.9 environment at: /Users/james/Documents/obi/code/obi-main/obi-one/.venv
Checked 4 packages in 14ms


CompletedProcess(args=['uv', 'pip', 'install', '--python', '/Users/james/Documents/obi/code/obi-main/obi-one/.venv/bin/python', 'neuroconv', 'pynwb', 'hdmf-zarr', 'aind-nwb-utils'], returncode=0)

## Clone the capsule and seed `data/`

The capsule needs the dispatcher's `job_*.json`. The job dict's `recording_dict` paths were generated by the dispatch script with `relative_to=../data` (relative to its `code/` cwd), which absolute-resolves to the toy recording on disk; the same relative path works from the NWB capsule's `data/` since both live at `/tmp/aind-ephys-*/data` (same depth).

In [3]:
nwb_repo = Path("/tmp/aind-ecephys-nwb")
if not nwb_repo.exists():
    subprocess.run(
        [
            "git", "clone", "--depth=1",
            "https://github.com/AllenNeuralDynamics/aind-ecephys-nwb.git",
            str(nwb_repo),
        ],
        check=True,
    )

data_dir = nwb_repo / "data"
results_dir = nwb_repo / "results"
data_dir.mkdir(exist_ok=True)
results_dir.mkdir(exist_ok=True)

for stale in list(data_dir.iterdir()) + list(results_dir.iterdir()):
    shutil.rmtree(stale) if stale.is_dir() else stale.unlink()

dispatch_results = Path.cwd() / "dispatch_results"
assert dispatch_results.exists(), "Run 1_aind_ephys_dispatch.ipynb first."

for entry in dispatch_results.iterdir():
    shutil.copy2(entry, data_dir / entry.name)

print("Seeded data dir:")
for p in sorted(data_dir.iterdir()):
    print(" ", p.name)

# Patch run_capsule.py for neuroconv API drift:
# - `add_electrodes_info_to_nwbfile` was renamed to `add_electrodes_to_nwbfile`.
capsule_py = nwb_repo / "code" / "run_capsule.py"
src = capsule_py.read_text()
patched = src.replace("add_electrodes_info_to_nwbfile", "add_electrodes_to_nwbfile")
if patched != src:
    capsule_py.write_text(patched)
    print("Patched neuroconv import: add_electrodes_info_to_nwbfile -> add_electrodes_to_nwbfile")

Cloning into '/tmp/aind-ecephys-nwb'...


Seeded data dir:
  job_0.json
Patched neuroconv import: add_electrodes_info_to_nwbfile -> add_electrodes_to_nwbfile


## Run the NWB-export capsule

`--write-raw` adds the raw `ElectricalSeries` to the NWB. `--skip-lfp` disables LFP-track writing. We use the HDF5 backend (`--backend hdf5`).

In [4]:
argv = [
    "python", "-u", "run_capsule.py",
    "--write-raw",
    "--skip-lfp",
    "--backend", "hdf5",
]
print("Running:", " ".join(argv))
result = subprocess.run(
    argv,
    cwd=nwb_repo / "code",
    capture_output=True,
    text=True,
    check=False,
)
print(result.stdout)
if result.returncode != 0:
    print("STDERR:\n", result.stderr)
    raise RuntimeError(f"NWB export failed with code {result.returncode}")

Running: python -u run_capsule.py --write-raw --skip-lfp --backend hdf5




NWB EXPORT ECEPHYS
Running NWB conversion with the following parameters:
Stub test: False
Stub seconds: 10.0
Write LFP: False
Write RAW: True
Temporal subsampling factor: 2
Spatial subsampling factor: 4
Highpass filter frequency: 0.1
Surface channel indices for agar probes: None
NWB backend: hdf5
Found 1 JSON job files
Session: session0
Number of NWB files to write: 1
Number of streams to write for each file: 1
Creating NWB file with info.
Processing block0_None_recording1
	Loading block0_None_recording1 from 1 JSON files
		block0_None_recording1
	Adding probe device: Probe from recording metadata
	Adding RAW data for stream None - segment 0
Added 1 streams
Configuring hdf5 backend
Writing NWB file to ../results/session0_block0_recording1.nwb
Writing time: 2.48s
Done writing ../results/session0_block0_recording1.nwb
NWB EXPORT ECEPHYS time: 2.53s



## Copy outputs next to the notebook

In [5]:
local_results_dir = Path.cwd() / "nwb_results"
if local_results_dir.exists():
    shutil.rmtree(local_results_dir)
shutil.copytree(results_dir, local_results_dir)

for p in sorted(local_results_dir.rglob("*")):
    rel = p.relative_to(local_results_dir)
    kind = "dir" if p.is_dir() else f"{p.stat().st_size} bytes"
    print(f"  {rel}  ({kind})")

  session0_block0_recording1.nwb  (79912624 bytes)


## Open the NWB file with pynwb and inspect contents

In [6]:
from pynwb import NWBHDF5IO

nwb_files = sorted(local_results_dir.glob("*.nwb"))
print("NWB files:", [p.name for p in nwb_files])

if nwb_files:
    with NWBHDF5IO(str(nwb_files[0]), "r") as io:
        nwb = io.read()
        print("session_id:        ", nwb.session_id)
        print("identifier:        ", nwb.identifier)
        print("session_start_time:", nwb.session_start_time)
        print("devices:           ", list(nwb.devices.keys()))
        print("electrode_groups:  ", list(nwb.electrode_groups.keys()))
        if nwb.electrodes is not None:
            print("electrodes:        ", len(nwb.electrodes), "rows")
        print("acquisition:       ", list(nwb.acquisition.keys()))
        for name, obj in nwb.acquisition.items():
            print(f"  {name}: shape={getattr(obj, 'data', None).shape if hasattr(obj, 'data') else 'n/a'}"
                  f" rate={getattr(obj, 'rate', 'n/a')}")

NWB files: ['session0_block0_recording1.nwb']
session_id:         session0
identifier:         8b195717-b450-4f15-9a5b-c675ff4e0dd3
session_start_time: 2026-04-29 13:29:48.157865+02:00
devices:            ['Probe']
electrode_groups:   ['Probe']
electrodes:         70 rows
acquisition:        ['ElectricalSeriesProbe']
  ElectricalSeriesProbe: shape=(300000, 70) rate=None
